In [46]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from scipy import stats
import matplotlib as plt
import warnings
warnings.filterwarnings('ignore')

In [47]:
DATA_PATH: str = "data.csv"

LAMBDA = 0.2
LOOKBACK_WINDOW = 20

STANDARDIZATION_METHODS = ['rank', "zscore"]

In [ ]:
def load_data(data_path: str = DATA_PATH) -> pd.DataFrame:
    df = pd.read_csv(data_path)

    df = df.dropna(subset=["date", "order_book_id"])

    df[df.columns.drop('pct_rate')] = df[df.columns.drop('pct_rate')].replace(0, np.nan)
    
    df['date'] = pd.to_datetime(df['date'])
    
    df = df.set_index(['date', 'order_book_id'])
    df = df.sort_index(level=['date', 'order_book_id'])
    return df

In [ ]:
def calculate_factor(df: pd.DataFrame, lam: float = LAMBDA, hist_wdw: int = LOOKBACK_WINDOW) -> pd.DataFrame:
    df_temp = df.swaplevel().sort_index()  # 暂时以股票做第一层以方便计算

    # 计算出过渡因子值
    df_temp['order_imbal'] = (df_temp['buy_value'] - df_temp['sell_value']) / (df_temp['buy_value'] + df_temp['sell_value'])
    df_temp['order_imbal'] = df_temp['order_imbal'].replace([np.inf, -np.inf], np.nan)

    def compute_group_factor(group: pd.DataFrame) -> pd.Series:
        order_imbals, returns = group['order_imbal'].values, group['pct_rate'].values
        out = np.full(len(group), np.nan)  # 默认返回Nan
        for i in range(hist_wdw - 1, len(group)):
            wdw_ord_imb, wdw_rets = order_imbals[i - hist_wdw + 1: i + 1], returns[i - hist_wdw + 1: i + 1]
            valid_rets_counts = np.sum(~np.isnan(wdw_rets))
            if valid_rets_counts < hist_wdw * 0.5: # Nan收益率值过多，代表性可能不足
                continue

            threshold = np.nanquantile(wdw_rets, lam)
            low_ret_mask = (wdw_rets <= threshold) & ~np.isnan(wdw_ord_imb)  # 低收益日中因子过渡值不能为Nan
            if low_ret_mask.any():
                out[i] = np.nanmean(wdw_ord_imb[low_ret_mask])

        return pd.Series(out, index=group.index)

    df_temp['factor'] = df_temp.groupby(level='order_book_id', group_keys=False).apply(compute_group_factor)

    df = df_temp.swaplevel().sort_index()  # 转变回数据集初始格式以便后续因子分析
    return df

In [ ]:
def standardize_factor(df: pd.DataFrame, method: str = 'rank') -> pd.DataFrame:
    if method not in STANDARDIZATION_METHODS:
        return df

    std_factor = 'factor_ranked' if method == 'rank' else 'factor_zscore'
    if method == 'rank':
        df[std_factor] = df['factor'].rank(method='average')
    else:
        median = df['factor'].median()
        mad = np.median(np.abs(df['factor'] - median))

        df[std_factor] = df['factor'].clip(lower = median - 2.5 * mad, upper = median + 2.5 * mad)
        
    df[std_factor] = (df[std_factor] - df[std_factor].mean()) / df[std_factor].std()

    return df

In [67]:
def get_residual(x: pd.DataFrame, factor_col: str) -> pd.Series:
    """
    获取因子值的买卖金额残差
    """
    x = x.dropna(subset=[factor_col, 'log_value'])
    if len(x) < 100:
        return pd.Series(np.nan, index=x.index)
    
    X = sm.add_constant(x['log_value'])
    y = x['factor']

    model = sm.OLS(y, X).fit()
    return pd.Series(model.resid, index=x.index)

def neutralize_factor(df: pd.DataFrame, factor: str = 'factor_ranked') -> pd.DataFrame:
    df = df.copy()
    df['log_value'] = np.log(df['buy_value'] + df['sell_value'])

    df[factor + "_neu"] = df.groupby('date').apply(lambda x: get_residual(x, factor)).reset_index(level=0, drop=True)

    df = df.drop(columns=['log_value'])
    return df

In [ ]:
def calculcate_ic_and_tstat(df: pd.DataFrame, factor_col: str = 'factor_ranked_neu') -> tuple[pd.Series, float]:
    ic_series = df.groupby('date').apply(lambda x: stats.spearmanr(x[factor_col], x['pct_rate'])[0]).dropna()

    ic_mean, ic_stdev = ic_series.mean(), ic_series.std()

    ir = ic_mean / ic_stdev if ic_stdev != 0 else  0

    tstat, p_val = stats.ttest_1samp(ic_series, 0)
    
    return ic_series, ic_mean, ic_stdev, ir, tstat, p_val

In [53]:
df = load_data()

In [ ]:
df = calculate_factor(df)

buy_volume     buy_value  sell_volume    sell_value  \
date       order_book_id                                                        
2020-01-02 000001.XSHE    84209533.0  1.414344e+09   68813654.0  1.156852e+09   
           000002.XSHE    50843277.0  1.679690e+09   50369763.0  1.662684e+09   
           000004.XSHE      705829.0  1.587629e+07    1079491.0  2.426869e+07   
           000005.XSHE     4971400.0  1.560739e+07    5442012.0  1.705402e+07   
           000006.XSHE     6488025.0  3.511604e+07    5987151.0  3.233682e+07   
...                              ...           ...          ...           ...   
2025-06-30 688799.XSHG     1406508.0  5.800410e+07    1767620.0  7.283469e+07   
           688800.XSHG     5851541.0  2.863658e+08    6998626.0  3.421055e+08   
           688819.XSHG     1788384.0  4.936817e+07    1576386.0  4.338212e+07   
           688981.XSHG    18172208.0  1.609141e+09   16180594.0  1.432188e+09   
           689009.XSHG     4815536.0  2.827421e+08    3667773.0  2.153004e+08   

                          pct_rate  order_imbal    factor  
date       order_book_id                                   
2020-01-02 000001.XSHE    0.025532     0.100145       NaN  
           000002.XSHE    0.011809     0.005088       NaN  
           000004.XSHE   -0.011510    -0.209052       NaN  
           000005.XSHE    0.016182    -0.044292       NaN  
           000006.XSHE    0.007463     0.041202       NaN  
...                            ...          ...       ...  
2025-06-30 688799.XSHG   -0.007451    -0.113350 -0.102069  
           688800.XSHG    0.006341    -0.088691 -0.101975  
           688819.XSHG    0.019838     0.064539 -0.068168  
           688981.XSHG    0.014848     0.058183 -0.063171  
           689009.XSHG    0.019644     0.135414 -0.126976  

[7065419 rows x 7 columns]

In [63]:
df = standardize_factor(df)

In [68]:
df = neutralize_factor(df)

In [69]:
df

buy_volume     buy_value  sell_volume    sell_value  \
date       order_book_id                                                        
2020-01-02 000001.XSHE    84209533.0  1.414344e+09   68813654.0  1.156852e+09   
           000002.XSHE    50843277.0  1.679690e+09   50369763.0  1.662684e+09   
           000004.XSHE      705829.0  1.587629e+07    1079491.0  2.426869e+07   
           000005.XSHE     4971400.0  1.560739e+07    5442012.0  1.705402e+07   
           000006.XSHE     6488025.0  3.511604e+07    5987151.0  3.233682e+07   
...                              ...           ...          ...           ...   
2025-06-30 688799.XSHG     1406508.0  5.800410e+07    1767620.0  7.283469e+07   
           688800.XSHG     5851541.0  2.863658e+08    6998626.0  3.421055e+08   
           688819.XSHG     1788384.0  4.936817e+07    1576386.0  4.338212e+07   
           688981.XSHG    18172208.0  1.609141e+09   16180594.0  1.432188e+09   
           689009.XSHG     4815536.0  2.827421e+08    3667773.0  2.153004e+08   

                          pct_rate  order_imbal    factor  factor_ranked  \
date       order_book_id                                                   
2020-01-02 000001.XSHE    0.025532     0.100145       NaN            NaN   
           000002.XSHE    0.011809     0.005088       NaN            NaN   
           000004.XSHE   -0.011510    -0.209052       NaN            NaN   
           000005.XSHE    0.016182    -0.044292       NaN            NaN   
           000006.XSHE    0.007463     0.041202       NaN            NaN   
...                            ...          ...       ...            ...   
2025-06-30 688799.XSHG   -0.007451    -0.113350 -0.102069       0.897397   
           688800.XSHG    0.006341    -0.088691 -0.101975       0.898731   
           688819.XSHG    0.019838     0.064539 -0.068168       1.310085   
           688981.XSHG    0.014848     0.058183 -0.063171       1.356926   
           689009.XSHG    0.019644     0.135414 -0.126976       0.506507   

                          factor_ranked_neu  
date       order_book_id                     
2020-01-02 000001.XSHE                  NaN  
           000002.XSHE                  NaN  
           000004.XSHE                  NaN  
           000005.XSHE                  NaN  
           000006.XSHE                  NaN  
...                                     ...  
2025-06-30 688799.XSHG             0.058845  
           688800.XSHG             0.023423  
           688819.XSHG             0.100531  
           688981.XSHG             0.026543  
           689009.XSHG             0.003686  

[7065419 rows x 9 columns]